In [6]:
from sentence_transformers import SentenceTransformer
import pandas as pd
from pprint import pprint
from datasets import load_dataset
import os
from dotenv import dotenv_values
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import PointStruct, Document
from groq import Groq

In [2]:
# Load environment variables
config = dotenv_values(".env")
HF_TOKEN  = config.get("HF_TOKEN")
QDRANT_CLOUD_API_KEY = config.get("QDRANT_CLOUD_API_KEY")
QDRANT_CLOUD_ENDPOINT = config.get("QDRANT_CLOUD_ENDPOINT")
GROQ_API_KEY  = config["GROQ_API_KEY"]

# Make HuggingFace token available to the transformers library
os.environ["HF_TOKEN"] = HF_TOKEN

In [4]:
#Embedding Model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [24]:
token = " هل يمكنني رؤية قائمة الطعام، من فضلك؟ "
vector = model.encode(token)
len(vector)

384

In [32]:
collection_name = "Morrocan_Chat_Culture"
# Connection With QDrant Cloud
client_qdrant = QdrantClient(
    url=QDRANT_CLOUD_ENDPOINT,
    api_key=QDRANT_CLOUD_API_KEY,
    cloud_inference=True
)

# Collection Creation
client_qdrant.create_collection(
    collection_name=collection_name,
    vectors_config={
        "text": models.VectorParams(size=384, distance=models.Distance.COSINE), # we use cosin similarity
    },
    sparse_vectors_config={"text-sparse": models.SparseVectorParams()},
)

True

In [31]:
client_qdrant.delete_collection(collection_name=collection_name)

True

In [ ]:
# Example Of Insert Dense Vector Into Qdrant Cloud
client_qdrant.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=1,
            vector={
                "text": vector
            },
            payload={
                "chunk": token,
            }

        )
    ]
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

In [37]:
# Connection Wth Groq
client_groq = Groq(api_key=GROQ_API_KEY)

In [ ]:
# Load Dataset From Hugging Face
ds = load_dataset(
    "atlasia/Atlaset",
    split="train",
    streaming=True # we do not download this dataset we create object refrence o dataset
)

In [ ]:
# Get Emebddings for Chunks
def get_embeddings(chunks):
    for chunk in chunks:
        chunk["embedding"] = model.encode(chunk["text"])

In [ ]:
# Add Chunks to Vectorial DataBase
def add_chunks_to_Qdrant(chunks):
    client_qdrant.upsert()
    pass

In [ ]:
# Get First 1000 rows and Store Them in Qdrant Cloud
total = 1000
number_of_chunks = 500

def get_rows(number_of_chunks):
    x = []
    for d in ds:
        print(d)
        x.append({"text": d, "embedding": []})
        number_of_chunks -= 10
        if (number_of_chunks < 0): break
        # Insert Chunks TO DB

    print(x)
get_rows(number_of_chunks)

NameError: name 'ds' is not defined

In [ ]:
# Check Similarity Between Question And Chunks
def get_relevent_chunks(embeddings_question):
    pass

In [ ]:
# Get hypothetical Embedding Documents

def get_llm_documents(question):
    """Generate a short hypothetical documentation passage for `question`."""
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role"   : "system",
                "content": (
                    f"You are a technical documentation writer. "
                    f"Write one clear, structured documentation for: {question}. "
                    f"Use simple English words, be brief, and stay on topic."
                ),
            }
        ],
        temperature=1,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = [chunk.choices[0].delta.content for chunk in completion]
    res = [s for s in res if s]
    return "".join(res)


In [ ]:
def get_hyde_documents_embeddings(hyde_documents):
    pass

In [ ]:
def search_by_cosin():
    pass

In [ ]:
# Get Reponse From LLM
def generate_response():
    pass

SyntaxError: incomplete input (3291812730.py, line 2)

In [ ]:
# Define Question
question = " ة كاملة و ف 1985 ?"

# Get Question Embedding
embeddings_question = model.encode(question)

# Get Relevent Chunks From Qdart
relevent_chunks = get_relevent_chunks(embeddings_question)

# Get hypothetical Embedding Documents
hyde_documents = get_llm_documents(question=question)

# Get Embedding for hyde documents
hyde_documents = get_hyde_documents_embeddings(hyde_documents=hyde_documents)

# Filtre To Get Context With Cosin Similarity
filtred_documents = search_by_cosin(relevent_chunks, hyde_documents)
context = filtred_documents

# Get Response
response = generate_response(question=question, context=filtred_documents)
print(response)

NameError: name 'model' is not defined

In [ ]:
# Hybrid Search

In [ ]:
# Hyde Architect

In [ ]:
# Get Relevent Documents


In [ ]:
# Get Context and Give LLM Context and Get Response